# P46 — Una imagen vale 16x16 palabras: Transformers para reconocimiento de imágenes a escala

## 1. Título y paper

**Paper:** *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*  
**Autoría:** Alexey Dosovitskiy, Lucas Beyer, Alexander Kolesnikov, y otros  
**Año y venue:** 2020 · arXiv:2010.11929 · ICLR 2021  
**Nivel:** L3 · **Motor:** `vit`  
**Ficha completa:** [`P46_vit`](../../papers/foundational/P46_vit/README.md)

**Hito:** Trata la imagen como una secuencia de parches y aplica un Transformer puro: la convolución deja de ser imprescindible en visión.

- [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La convolución traía de fábrica localidad y equivarianza a la traslación, y se asumía que sin esos sesgos inductivos la visión no funcionaría.
2. Ejecutar una implementación mínima de la propuesta: Partir la imagen en parches, proyectarlos como si fueran tokens, añadir codificación posicional y aplicar el encoder del Transformer sin más.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P04
- P08


## 4. Intuición

Trocea la imagen en cuadraditos, ponlos en fila y trátalos como si fueran palabras de una frase. Suena absurdo —se pierde toda la noción de vecindad— y funciona, si tienes datos suficientes.


## 5. Concepto mínimo

```text
imagen 224×224  →  parches 16×16  →  (224/16)² = 196 tokens  (+1 de clase)

cada parche: 16·16·3 = 768 valores → proyección lineal → token
```

A partir de ahí, es el encoder de [P08](../../papers/foundational/P08_transformer/README.md) sin ninguna modificación.


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('vit', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos tokens salen de una imagen 224×224 con parches de 16?
2. ¿Qué pasa con el coste si bajas el parche a 8?
3. ¿Qué sesgo inductivo pierde respecto a una CNN?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('vit', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('vit', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Una imagen se convierte en una secuencia de ~200 tokens: exactamente el mismo problema que una frase. Y bajar el tamaño de parche multiplica el coste **al cuadrado**, porque la atención es cuadrática en el número de tokens.


## 10. Comentario pedagógico

El resultado del paper viene con una condición grande: **funciona si se preentrena con muchísimos datos**. Con conjuntos medianos, la CNN gana, porque su sesgo inductivo vale más que la flexibilidad. Es un ejemplo limpio del compromiso sesgo/datos.


## 11. Error o anti-patrón deliberado

Anti-patrón: citar ViT como «los Transformers superan a las CNN» sin la condición de datos.


In [ ]:
for datos in ('1M imagenes', '14M imagenes', '300M imagenes'):
    ganador = 'CNN' if '1M' in datos else 'ViT' if '300M' in datos else 'depende'
    print(f'preentrenado con {datos:<15} → suele ganar: {ganador}')

## 12. Corrección

El enunciado correcto incluye el régimen:


In [ ]:
enunciado = {'valido': 'con preentrenamiento a gran escala, ViT iguala o supera a CNN comparables',
             'no_valido': 'los Transformers son mejores que las CNN en vision',
             'razon': 'sin sesgo inductivo hace falta mas dato para aprender lo mismo',
             'consecuencia': 'la eleccion depende de tu regimen de datos, no de la moda'}
show(enunciado)

## 13. Desafío guiado

Calcula los tokens y el coste relativo de una imagen 512×512 con parches de 8.


In [ ]:
r = run_paper_lab('vit', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena un ViT pequeño y una CNN de parámetros comparables sobre un conjunto de 10 000 imágenes. Repite con aumento de datos agresivo y comprueba si la brecha se cierra.


## 15. Evidencia de aprendizaje

Guarda la tabla de tokens y coste, y tu enunciado con la condición de régimen de datos.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P46_vit/README.md) · evaluación formal: [`assessments/papers/P46_vit.md`](../../assessments/papers/P46_vit.md)


## 16. Cierre

El mismo bloque sirve para texto e imagen. Y también para problemas científicos que llevaban décadas abiertos.


## 17. Conexión con el siguiente hito

- P18
- modelos multimodales

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
